# FERAttentionLite — FER-2013 Training

A compact facial-expression recognition model trained **from scratch** on FER-2013.

**Reproduced result**
- Validation accuracy: **67.51%**
- Validation macro F1: **0.6577**
- PrivateTest accuracy: **68.38%**
- PrivateTest macro F1: **0.6664**
- Trainable parameters: **1,211,033**

A separate run of the same recipe reached **68.43% accuracy / 0.6690 macro F1**.

Architecture: `48×48 grayscale → CNN → channel attention → spatial attention → GAP → 7 classes`.

In [ ]:
# 1. Environment
!nvidia-smi

import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

In [ ]:
# 2. Configuration
SEED = 42
NUM_CLASSES = 7
BATCH_SIZE = 128
EPOCHS = 110
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02
NUM_WORKERS = 0

FER_LABELS = [
    "Angry", "Disgust", "Fear", "Happy",
    "Sad", "Surprise", "Neutral",
]

CSV_PATH = "/kaggle/input/fer2013/fer2013.csv"
OUTPUT_DIR = Path("/kaggle/working/fer_attention_lite")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)

if not Path(CSV_PATH).exists():
    matches = list(Path("/kaggle/input").rglob("fer2013.csv"))
    if not matches:
        raise FileNotFoundError("Add a FER-2013 dataset containing fer2013.csv to this Kaggle notebook.")
    CSV_PATH = str(matches[0])

print("Dataset:", CSV_PATH)

## Dataset

The notebook expects the common FER-2013 CSV format:

```text
emotion,pixels,Usage
0,"70 80 82 ...",Training
...
```

The standard split names are `Training`, `PublicTest`, and `PrivateTest`.

In [ ]:
# 3. Load and inspect FER-2013
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
display(df.head())
display(df["Usage"].value_counts())

class_counts = df["emotion"].value_counts().sort_index()
class_table = pd.DataFrame({
    "emotion_id": range(NUM_CLASSES),
    "label": FER_LABELS,
    "count": [class_counts.get(i, 0) for i in range(NUM_CLASSES)],
})
display(class_table)

plt.figure(figsize=(10, 4))
plt.bar(FER_LABELS, class_table["count"])
plt.title("FER-2013 class distribution")
plt.ylabel("Images")
plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.25)
plt.show()

In [ ]:
# 4. Dataset and augmentation
class FER2013Dataset(Dataset):
    def __init__(self, dataframe, split, transform=None):
        self.df = dataframe[dataframe["Usage"] == split].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        pixels = np.fromstring(row["pixels"], dtype=np.uint8, sep=" ").reshape(48, 48)
        image = transforms.functional.to_pil_image(pixels)
        label = int(row["emotion"])

        if self.transform:
            image = self.transform(image)

        return image, label

train_tfms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(7),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.97, 1.03),
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

eval_tfms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_ds = FER2013Dataset(df, "Training", train_tfms)
val_ds = FER2013Dataset(df, "PublicTest", eval_tfms)
test_ds = FER2013Dataset(df, "PrivateTest", eval_tfms)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

## FERAttentionLite

The model is deliberately small and does not use a pretrained backbone.

- convolutional blocks learn local facial features;
- channel attention reweights feature maps;
- spatial attention reweights facial locations;
- global average pooling keeps the classifier compact.

In [ ]:
# 5. Model
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return x * self.mlp(self.pool(x))

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size // 2, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        avg_map = torch.mean(x, dim=1, keepdim=True)
        max_map, _ = torch.max(x, dim=1, keepdim=True)
        weight = self.attn(torch.cat([avg_map, max_map], dim=1))
        return x * weight

class FERAttentionLite(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.block1 = nn.Sequential(
            ConvBNReLU(1, 32),
            ConvBNReLU(32, 32),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.10),
        )
        self.block2 = nn.Sequential(
            ConvBNReLU(32, 64),
            ConvBNReLU(64, 64),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.15),
        )
        self.block3_pre = nn.Sequential(
            ConvBNReLU(64, 128),
            ConvBNReLU(128, 128),
        )
        self.channel_attention = ChannelAttention(128, reduction=8)
        self.spatial_attention = SpatialAttention(kernel_size=7)
        self.block3_post = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Dropout2d(0.20),
        )
        self.block4 = nn.Sequential(
            ConvBNReLU(128, 256),
            ConvBNReLU(256, 256),
            nn.Dropout2d(0.25),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3_pre(x)
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        x = self.block3_post(x)
        x = self.block4(x)
        x = self.gap(x)
        return self.classifier(x)

model = FERAttentionLite(NUM_CLASSES).to(DEVICE)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

dummy = torch.randn(4, 1, 48, 48, device=DEVICE)
with torch.no_grad():
    output = model(dummy)

print("Output shape:", tuple(output.shape))
print("Trainable parameters:", f"{num_params:,}")

In [ ]:
# 6. Soft class weights, loss, optimizer, scheduler
train_labels = train_ds.df["emotion"].to_numpy()
counts = np.bincount(train_labels, minlength=NUM_CLASSES)

weights = np.sqrt(counts.sum() / (NUM_CLASSES * np.maximum(counts, 1)))
weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

display(pd.DataFrame({
    "class": FER_LABELS,
    "count": counts,
    "soft_weight": weights,
}))

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6,
)

use_amp = DEVICE.type == "cuda"
try:
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    def amp_context():
        return torch.amp.autocast("cuda", enabled=use_amp)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    def amp_context():
        return torch.cuda.amp.autocast(enabled=use_amp)

In [ ]:
# 7. Evaluation helper
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_y, all_pred, all_probs = [], [], []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)
        loss = nn.functional.cross_entropy(logits, y)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        total_loss += loss.item() * x.size(0)
        all_y.extend(y.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_y, all_pred),
        "macro_f1": f1_score(all_y, all_pred, average="macro"),
        "y_true": np.array(all_y),
        "y_pred": np.array(all_pred),
        "probs": np.array(all_probs),
    }

In [ ]:
# 8. Train
history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": [],
    "lr": [],
}

best_f1 = -1.0
best_epoch = -1

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    seen = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{EPOCHS}")

    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with amp_context():
            logits = model(x)
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * x.size(0)
        seen += x.size(0)
        pbar.set_postfix(loss=f"{running_loss / seen:.4f}")

    scheduler.step()
    train_loss = running_loss / seen
    val_metrics = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_metrics["loss"])
    history["val_accuracy"].append(val_metrics["accuracy"])
    history["val_macro_f1"].append(val_metrics["macro_f1"])
    history["lr"].append(optimizer.param_groups[0]["lr"])

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_acc={val_metrics['accuracy']*100:.2f}% | "
        f"val_f1={val_metrics['macro_f1']:.4f}"
    )

    if val_metrics["macro_f1"] > best_f1:
        best_f1 = val_metrics["macro_f1"]
        best_epoch = epoch
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "labels": FER_LABELS,
            "input_size": [1, 48, 48],
            "model_name": "FERAttentionLite",
        }, OUTPUT_DIR / "best.pt")
        print("  saved best.pt")

    with open(OUTPUT_DIR / "history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

print("Best epoch:", best_epoch)
print("Best validation Macro F1:", best_f1)

In [ ]:
# 9. Learning curves
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, np.array(history["val_accuracy"]) * 100, label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history["val_macro_f1"], label="Validation macro F1")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

In [ ]:
# 10. Final PrivateTest evaluation
checkpoint = torch.load(OUTPUT_DIR / "best.pt", map_location=DEVICE)
model.load_state_dict(checkpoint["model_state"])
model.eval()

test_metrics = evaluate(model, test_loader)

print(f"Best epoch: {checkpoint['epoch']}")
print(f"Validation Accuracy: {checkpoint['val_accuracy']*100:.2f}%")
print(f"Validation Macro F1: {checkpoint['val_macro_f1']:.4f}")
print(f"PrivateTest Accuracy: {test_metrics['accuracy']*100:.2f}%")
print(f"PrivateTest Macro F1: {test_metrics['macro_f1']:.4f}")

print("\nClassification report:\n")
print(classification_report(
    test_metrics["y_true"],
    test_metrics["y_pred"],
    target_names=FER_LABELS,
    digits=4,
))

In [ ]:
# 11. Confusion matrix
cm = confusion_matrix(test_metrics["y_true"], test_metrics["y_pred"])

fig, ax = plt.subplots(figsize=(9, 9))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=FER_LABELS,
)
disp.plot(
    ax=ax,
    xticks_rotation=45,
    cmap="Blues",
    values_format="d",
    colorbar=False,
)
plt.title("FERAttentionLite — PrivateTest Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# 12. High-confidence errors
y_true = test_metrics["y_true"]
y_pred = test_metrics["y_pred"]
probs = test_metrics["probs"]

wrong_idx = np.where(y_true != y_pred)[0]
wrong_conf = probs[wrong_idx, y_pred[wrong_idx]]
selected = wrong_idx[np.argsort(-wrong_conf)[:20]]

plt.figure(figsize=(15, 12))

for j, idx in enumerate(selected):
    image_tensor, true_label = test_ds[idx]
    image = image_tensor[0].numpy() * 0.5 + 0.5
    pred_label = y_pred[idx]
    confidence = probs[idx, pred_label]

    plt.subplot(4, 5, j + 1)
    plt.imshow(image, cmap="gray")
    plt.title(
        f"T: {FER_LABELS[true_label]}\n"
        f"P: {FER_LABELS[pred_label]} ({confidence:.2f})"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 13. Export summary
summary = {
    "model": "FERAttentionLite",
    "best_epoch": int(checkpoint["epoch"]),
    "val_accuracy": float(checkpoint["val_accuracy"]),
    "val_macro_f1": float(checkpoint["val_macro_f1"]),
    "test_accuracy": float(test_metrics["accuracy"]),
    "test_macro_f1": float(test_metrics["macro_f1"]),
    "trainable_parameters": int(num_params),
    "labels": FER_LABELS,
}

with open(OUTPUT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nCheckpoint:", OUTPUT_DIR / "best.pt")

## Reproduced result

The repository's reproduced training run achieved **68.38% PrivateTest accuracy** and **0.6664 macro F1** with **1,211,033 trainable parameters**. A separate run of the same recipe reached **68.43% / 0.6690**, so the observed operating point is consistently around 68.4%.

For real-time CPU inference, export the selected checkpoint and use the modular pipeline in the repository.